1. Fetch the artifact we just created (sample.csv) from W&B and read it with pandas:

In [7]:
import wandb
import pandas as pd
import os
import tempfile

# Note that we use save_code=True in the call to wandb.init so the notebook is uploaded and versioned by W&B
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)

# Download the artifact with a temporary cache directory to avoid Windows path issues
with tempfile.TemporaryDirectory() as tmpdir:
    try:
        artifact = wandb.use_artifact("sample.csv:latest")
        local_path = artifact.download(root=tmpdir)
        # Find the actual file in the downloaded directory
        for root_dir, dirs, files in os.walk(local_path):
            for file in files:
                if file.endswith('.csv'):
                    df = pd.read_csv(os.path.join(root_dir, file))
                    break
        else:
            raise FileNotFoundError("No CSV file found in artifact")
    except Exception as e:
        print(f"Error downloading artifact: {e}")
        print("Attempting to read from local file...")
        local_file = os.path.join(os.path.dirname(os.getcwd()), "components", "get_data", "data", "sample.csv")
        local_file_alt = os.path.join(os.getcwd(), "..", "..", "components", "get_data", "data", "sample1.csv")
        
        if os.path.exists(local_file):
            print(f"Using local file: {local_file}")
            df = pd.read_csv(local_file)
        elif os.path.exists(local_file_alt):
            print(f"Using local file: {local_file_alt}")
            df = pd.read_csv(local_file_alt)
        else:
            print("ERROR: Could not find sample.csv artifact or local file")
            print("Please ensure the MLflow pipeline's get_data step has completed.")
            raise RuntimeError("sample.csv not available")

wandb:   1 of 1 files downloaded.  


Error downloading artifact: No CSV file found in artifact
Attempting to read from local file...
Using local file: c:\Users\mgent\Documents\WGU\Semester 4\501 - DevOps ML\Project 1\Project-Build-an-ML-Pipeline-Starter\src\eda\..\..\components\get_data\data\sample1.csv

Attempting to read from local file...
Using local file: c:\Users\mgent\Documents\WGU\Semester 4\501 - DevOps ML\Project 1\Project-Build-an-ML-Pipeline-Starter\src\eda\..\..\components\get_data\data\sample1.csv


2. Explore the data in df

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              20000 non-null  int64  
 1   name                            19993 non-null  object 
 2   host_id                         20000 non-null  int64  
 3   host_name                       19992 non-null  object 
 4   neighbourhood_group             20000 non-null  object 
 5   neighbourhood                   20000 non-null  object 
 6   latitude                        20000 non-null  float64
 7   longitude                       20000 non-null  float64
 8   room_type                       20000 non-null  object 
 9   price                           20000 non-null  int64  
 10  minimum_nights                  20000 non-null  int64  
 11  number_of_reviews               20000 non-null  int64  
 12  last_review                     

In [9]:
df.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,2.000000e+04,2.000000e+04,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,15877.000000,20000.000000,20000.000000
mean,1.892380e+07,6.746034e+07,40.728455,-73.952125,153.269050,6.992100,23.274100,1.377446,6.955450,112.901200
std,1.101223e+07,7.857936e+07,0.054755,0.046559,243.325609,21.645449,44.927793,1.683006,32.433831,131.762226
min,2.539000e+03,2.571000e+03,40.508730,-74.239140,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.393540e+06,7.853718e+06,40.689420,-73.983030,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.952117e+07,3.111431e+07,40.722730,-73.955640,105.000000,2.000000,5.000000,0.720000,1.000000,44.000000
75%,2.912936e+07,1.068426e+08,40.762990,-73.936380,175.000000,5.000000,23.000000,2.010000,2.000000,229.000000
max,3.648561e+07,2.742733e+08,40.913060,-73.717950,10000.000000,1250.000000,607.000000,27.950000,327.000000,365.000000


In [ ]:
df.head()

3. What do you notice in the data? Look around and see what you can find.

> For example, there are missing values in a few columns and the column `last_review` is a date but it is in string format. Look also at the `price` column, and note the outliers. There are some zeros and some very high prices. After talking to your stakeholders, you decide to consider from a minimum of `$10` to a maximum of `$350` per night.

4. Fix some of the little problems we have found in the data with the following code:

In [ ]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()
# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

Note how we did not impute missing values. We will do that in the inference pipeline, so we will be able to handle missing values also in production.

5. Check with df.info() that all obvious problems have been solved

In [ ]:
df.info()

6. Terminate the run by running `run.finish()`

In [ ]:
run.finish()

7. Save the notebook.